In [ ]:
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import matplotlib.patches as mpatches


def set_publication_style():
    plt.style.use('default')

    # My custom palette
    publication_settings = {
        "font.family": "sans-serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "Computer Modern"],
        "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
        "mathtext.fontset": "stix",

        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "grid.color": "lightgray",

        "font.size": 8,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 12,

        "figure.dpi": 130,
        "savefig.dpi": 300,
        "savefig.bbox": 'tight',
        "savefig.format": 'pdf',
        "savefig.transparent": False,

        "lines.linewidth": 1.5,
        "lines.markersize": 6,
        "scatter.marker": "o",

        "axes.linewidth": 0.8,
        "axes.grid": False,
        "grid.linewidth": 0.5,
        "grid.alpha": 0.3,

        "image.cmap": "RdBu",

        "legend.frameon": True,
        "legend.framealpha": 0.8,
        "legend.edgecolor": "black",

        "errorbar.capsize": 3,
    }

    plt.rcParams.update(publication_settings)

set_publication_style()

In [ ]:
"""
    (I - P) rho = S
    P     transmission portion
    S     injection
    rho   flux carried by narrator
"""

import re
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import scipy.sparse as sp
import scipy.sparse.linalg as spla


# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
SCHOLARS_PATH = 'all_narrators_global.json'

HADITH_PATH   = 'bukhari.json'
# HADITH_PATH   = 'muslim.json'
# HADITH_PATH   = 'abudawud.json'
# HADITH_PATH   = 'nasai.json'

GEN_CAP       = 4
PEER_GAP_MAX  = 20.0
MAX_GEN       = 12
WEIGHT_MODE   = 'occurrence'
MAX_HOPS      = 12

_STYLE = {'axes.spines.top': False, 'axes.spines.right': False}

# ----------------------------------------------------------------------
# GRAPH BUILDER
# ----------------------------------------------------------------------


# We do some cleaning below, death should be ignored though since its not used any most cases not recorded at all
def build_metadata(path=SCHOLARS_PATH):
    meta = pd.read_json(path)
    meta['narrator_id'] = meta['narrator_id'].astype(str)
    meta = meta.drop_duplicates('narrator_id', keep='first')

    def gen_rank(s):
        if s is None or (isinstance(s, float) and np.isnan(s)): return None
        m = re.match(r'^\s*(\d+)', str(s).strip())
        if not m: return None
        k = int(m.group(1))
        return k - 1 if 1 <= k <= MAX_GEN else None

    def death_ah(s):
        if not isinstance(s, str): return np.nan
        nums = [int(x) for x in re.findall(r'\d+', s)]
        if not nums: return np.nan
        ah = [x for x in nums if 0 < x <= 400]
        return float(ah[0]) if ah else float(nums[0])

    meta['gen_rank'] = meta['generation_en'].map(gen_rank)
    meta['death_ah'] = meta['death_year_en'].map(death_ah)
    return meta

# We create the narrator chains by parsing the indices in each hadith
def parse_segments(cell):
    if cell is None or (isinstance(cell, float) and np.isnan(cell)): return []
    if not isinstance(cell, dict): raise TypeError(f"Expected dict, got {type(cell).__name__}")
    segments, cur = [], []
    for x in cell.get('scholar_indices') or []:
        if x is None or x == '' or (isinstance(x, float) and np.isnan(x)):
            if len(cur) > 1: segments.append(cur[::-1])
            cur = []
        else: cur.append(str(x))
    if len(cur) > 1: segments.append(cur[::-1])
    return segments

# Orientation might be redundant,but just in case, also sometimes in the same hadith two different chains are mentioned which trips the code
def load_chains(meta, path=HADITH_PATH):
    df = pd.read_json(path)
    df['segments'] = df['transmission_information'].apply(parse_segments)
    hid_col = next((c for c in ('hadith_id', 'id', 'hadith_no', 'reference') if c in df.columns), None)
    if hid_col is None:
        df = df.reset_index().rename(columns={'index': 'auto_hid'})
        hid_col = 'auto_hid'

    gmap = meta.set_index('narrator_id')['gen_rank'].dropna().to_dict()
    inc = dec = 0
    for segs in df['segments']:
        for seg in segs:
            g = [gmap[x] for x in seg if x in gmap]
            if len(g) >= 2:
                inc += g[0] < g[-1]
                dec += g[0] > g[-1]
    orient = dict(increasing=inc, decreasing=dec, reversed=True, margin=abs(inc - dec) / max(inc + dec, 1))
    if dec > inc: raise ValueError(f"orientation assumption violated")
    return df, hid_col, orient


# Actual transmissions
def build_edges(df, hid_col):
    rows, terminal_w, n_hops = [], Counter(), []
    for r in df.itertuples(index=False):
        hid = getattr(r, hid_col)
        for seg in r.segments:
            for p, (a, b) in enumerate(zip(seg, seg[1:])):
                if a != b:
                    rows.append({'hadith_id': hid, 'pos': p, 'teacher': a,
                                 'student': b, 'chain_len': len(seg)})
            terminal_w[seg[-1]] += 1
            n_hops.append(len(seg))

    edges_long = pd.DataFrame(rows)
    w = edges_long.groupby(['teacher', 'student']).size()
    edge_weights = Counter({(t, s): int(c) for (t, s), c in w.items()})
    return edges_long, edge_weights, terminal_w, float(np.mean(n_hops))

# Create a directed-graph, as before death is redundant
def build_graph(edge_weights, meta):
    G = nx.Graph()
    for (a, b), w in edge_weights.items():
        if G.has_edge(a, b): G[a][b]['weight'] += w
        else: G.add_edge(a, b, weight=w)
    nodes = list(G.nodes())
    idx = {v: i for i, v in enumerate(nodes)}
    gmap = meta.set_index('narrator_id')['gen_rank']
    dmap = meta.set_index('narrator_id')['death_ah']
    gen   = np.array([gmap.get(v, np.nan) for v in nodes], dtype=float)
    death = np.array([dmap.get(v, np.nan) for v in nodes], dtype=float)
    return G, nodes, idx, gen, death


# Only vertical gen1 -> gen2 transmission, single generation transmission breaks the acyclic structure, if death dates were correct -
# - it would gratly simplify the process, since we determine generation by maximum and minimum death year for some cohor,
def _classify(ia, ib, gen):
    ga, gb = gen[ia], gen[ib]
    if np.isnan(ga) or np.isnan(gb):
        return 'no_generation', None, None
    if ga == gb:
        return 'lateral_same_gen', None, None
    teacher, student = (ia, ib) if ga < gb else (ib, ia)
    if gen[student] - gen[teacher] > GEN_CAP:
        return 'over_gen_cap', None, None
    return 'kept_by_generation', teacher, student


# Actually drop the lateral edges
def split_edges(edge_weights, idx, gen, death):
    n = len(idx)
    vr, vc, vd = [], [], []
    mass = Counter()
    for (a, b), w in edge_weights.items():
        verdict, teacher, student = _classify(idx[a], idx[b], gen)
        mass[verdict] += w
        if teacher is not None:
            vr.append(student); vc.append(teacher); vd.append(w)
    A_vert = sp.csr_matrix((vd, (vr, vc)), shape=(n, n), dtype=float)
    A_vert.sum_duplicates()
    drops = dict(
        lateral=mass['lateral_same_gen'],
        no_generation=mass['no_generation'],
        over_gen_cap=mass['over_gen_cap'],
    )
    return A_vert, drops


# We create the absorbing OMEGA row here, and append n + 1 for the new dimensions, and normalization and flag orphans, which are just
# narrators with zero column sum, meaning transmission never reached the end
def build_operators(A_vert, nodes, idx, terminal_w):
    n = len(nodes)
    OMEGA = n
    c = A_vert.tocoo()
    r_, c_, d_ = list(c.row), list(c.col), list(c.data)
    terminal_placed = 0.0
    for v, w in terminal_w.items():
        if v in idx:
            r_.append(OMEGA); c_.append(idx[v]); d_.append(float(w))
            terminal_placed += w
    A_aug = sp.csr_matrix((d_, (r_, c_)), shape=(n + 1, n + 1), dtype=float)
    A_aug.sum_duplicates()
    col = np.asarray(A_aug.sum(axis=0)).ravel()
    orphan = (col == 0)
    orphan[OMEGA] = False
    P = A_aug.multiply(1.0 / np.where(col == 0, 1.0, col)).tocsr()
    return P, OMEGA, orphan, terminal_placed

def assert_acyclic(A_vert):
    c = A_vert.tocoo()
    D = nx.DiGraph()
    D.add_nodes_from(range(A_vert.shape[0]))
    D.add_edges_from(zip(c.col.tolist(), c.row.tolist()))
    if not nx.is_directed_acyclic_graph(D):
        raise ValueError("Transmission graph contains a cycle.")
    return nx.dag_longest_path_length(D)


# Solvers
def make_source(A_vert, gen, seeding='uniform'):
    """
    Mathematical Formulation:
      Roots (R): R = {u in V : (A^T 1)_u > 0, (A 1)_u = 0}
      Source Vector (S): S_u = 1 (if uniform) OR (A^T 1)_u (if out-strength)
    """
    in_s  = np.asarray(A_vert.sum(axis=1)).ravel()  # In-degree / incoming edge sum: (A 1)_u
    out_s = np.asarray(A_vert.sum(axis=0)).ravel() # Out-degree / outgoing edge sum: (A^T 1)_u
    roots = np.where((in_s == 0) & (out_s > 0))[0]  # Isolate nodes with 0 incoming and >0 outgoing edges

    S = np.zeros(len(gen) + 1)
    S[roots] = 1.0 if seeding == 'uniform' else out_s[roots]
    return S, roots


# Just solves the original flux
def solve_flux(P, S):
    """
      Original Equation: (I - P) rho = S
      Solution via Fundamental Matrix: rho = (I - P)^(-1) S
    """
    N = P.shape[0]
    return spla.spsolve((sp.eye(N, format='csc') - P.tocsc()), S)

# Solves backward and appends +1 for each node, basically a counter, although the results converge later
def downstream_depth(P):
    """
      Depth Recursive Equation: nu = 1 + P^T nu
      Rearranged Linear System: (I - P^T) nu = 1
    """
    N = P.shape[0]
    # Solves backward from the collector using P transpose (upper triangular)
    return spla.spsolve((sp.eye(N, format='csc') - P.T).tocsc(), np.ones(N))


# This can be said to model the evolution of flux over hops, and the actual final result over ||S|| can be said to be the average chain length
def path_decomposition(P, S, max_k=MAX_HOPS):
    """
      Hop Decomposition Series: rho = sum_{k=0}^m R_k
      Iterative Step: R_k = P * R_{k-1}  (where R_0 = S)
    """
    out, v = [], S.copy()
    while v.sum() > 1e-12:
        if len(out) >= max_k: raise RuntimeError("Did not drain; check acyclicity")
        out.append(v.copy())
        v = P.dot(v)  # Advances material forward by one hop: P^k S
    return np.vstack(out)


# Simply how the system conserves flow, except the orphans which are leaks and OMEGA
def edge_flux(P, rho_full, nodes, name_map):
    """
      Kirchhoff:  / Edge Flux: f_{vu} = P_{vu} * rho_u
      Ensures local conservation: sum_v f_{vu} = rho_u
    """
    F = P.multiply(rho_full[np.newaxis, :]).tocoo()
    n = len(nodes)
    lbl = lambda i: 'COLLECTOR' if i >= n else name_map.get(nodes[i], '?')
    return pd.DataFrame({
        'teacher': [lbl(j) for j in F.col],
        'student': [lbl(i) for i in F.row],
        'flux': F.data,
    }).sort_values('flux', ascending=False)


def build_model(seeding='uniform', path=HADITH_PATH, df=None, meta=None):
    meta = build_metadata() if meta is None else meta
    if df is None:
        df, hid_col, orient = load_chains(meta, path)
    else:
        hid_col = next(c for c in ('hadith_id', 'id', 'hadith_no', 'reference', 'auto_hid')
                       if c in df.columns)
    edges_long, edge_weights, terminal_w, emp_hops = build_edges(df, hid_col)
    G, nodes, idx, gen, death = build_graph(edge_weights, meta)
    A_vert, drops = split_edges(edge_weights, idx, gen, death)
    longest = assert_acyclic(A_vert)

    P, OMEGA, orphan, terminal_placed = build_operators(A_vert, nodes, idx, terminal_w)
    S, roots = make_source(A_vert, gen, seeding)

    n = len(nodes)
    rho_full = solve_flux(P, S)
    rho, absorbed = rho_full[:n], rho_full[OMEGA]
    RK = path_decomposition(P, S)

    # conservation checks
    stranded = float(rho_full[orphan].sum())
    resid = abs(S.sum() - absorbed - stranded)
    assert resid < 1e-8 * max(S.sum(), 1.0), f"conservation violated: {resid:.3e}"

    nu = downstream_depth(P)
    nu_nodes = np.where(orphan[:n], np.nan, nu[:n])
    mean_len = float(nu @ S) / S.sum()

    name_map = meta.set_index('narrator_id')['name_en']
    kk = np.arange(RK.shape[0])[:, None]
    mass = RK[:, :n].sum(axis=0)
    mean_hops = np.where(mass > 1e-15, (RK[:, :n] * kk).sum(axis=0) / np.where(mass > 1e-15, mass, 1.0), np.nan)

    nodes_df = pd.DataFrame({
        'narrator_id': nodes, 'name_en': [name_map.get(v, '?') for v in nodes],
        'gen': gen, 'death_ah': death, 'flux': rho, 'flux_share': rho / S.sum(),
        'nu_depth': nu_nodes, 'mean_hops': mean_hops, 'orphan': orphan[:n],
    })

    w_vert = float(A_vert.sum())
    total_recorded = w_vert + drops['lateral'] + drops['no_generation'] + drops['over_gen_cap']
    root_gens = gen[roots]

    diag = dict(
        n_nodes=n, n_vertical_edges=int(A_vert.nnz), longest_chain=longest,
        n_roots=len(roots), injected=float(S.sum()),
        absorbed=float(absorbed), stranded=stranded,
        orphan_share=stranded / S.sum() if S.sum() else np.nan,
        unreached=int((rho <= 1e-15).sum()),
        # preprocessing drops, all over a common denominator
        total_recorded=total_recorded,
        lateral_share=drops['lateral'] / total_recorded,
        over_gen_cap_share=drops['over_gen_cap'] / total_recorded,
        no_generation_share=drops['no_generation'] / total_recorded,
        kept_share=w_vert / total_recorded,
        gen_cap=GEN_CAP,
        # root composition
        roots_gen0=int(np.nansum(root_gens == 0)),
        roots_gen_max=float(np.nanmax(root_gens)) if len(root_gens) else np.nan,
        roots_no_gen=int(np.isnan(root_gens).sum()),
        mean_chain_len_model=mean_len - 1.0, mean_chain_len_empirical=emp_hops,
    )

    return dict(meta=meta, hadith=df, P=P, OMEGA=OMEGA, orphan=orphan, S=S, roots=roots,
                rho=rho, rho_full=rho_full, absorbed=absorbed, nu=nu, RK=RK,
                nodes_df=nodes_df, name_map=name_map, diag=diag, gen=gen, nodes=nodes,
                A_vert=A_vert, terminal_w=terminal_w, idx=idx, G=G,
                edges_long=edges_long, edge_weights=edge_weights)


# Analysis

# Gets generation layers
def _layers(M):
    gen = M['gen']
    levels = sorted(set(gen[~np.isnan(gen)].astype(int)))
    return {t: np.where(gen == t)[0] for t in levels}, levels

# Here we zero out a certain narrator by DPD, since the system is nillpotent it will zero out all tranmissions and received information,
# then we compare the actual rho to the top narrators, and we compare by simply removing one and the preceedings ones as well, the result is
# interesting cause it can quantify redundancy, meaning how much shared/identical flow are narrators carrying
def ablation(M, K=12, abs_max=0.50, pool_size=60):
    P, OMEGA, n = M['P'], M['OMEGA'], len(M['nodes'])
    N, S = P.shape[0], M['S']
    I = sp.eye(N, format='csc')

    def delivered(rm):
        keep = np.ones(N)
        keep[list(rm)] = 0.0
        D = sp.diags(keep)
        return spla.spsolve(I - (D @ P @ D).tocsc(), S * keep)[OMEGA]

    base = delivered([])
    onward = np.asarray(M['A_vert'].sum(axis=0)).ravel()
    absorb = np.array([M['terminal_w'].get(v, 0) for v in M['nodes']], float)
    abs_frac = absorb / np.maximum(absorb + onward, 1e-12)

    root_set = set(int(r) for r in M['roots'])
    elig = np.array([i for i in range(n)
                     if i not in root_set and abs_frac[i] < abs_max
                     and M['rho'][i] > 1e-12])
    order = [int(x) for x in elig[np.argsort(-M['rho'][elig])]]

    curve = pd.DataFrame([{
        'k': k + 1,
        'i': order[k],
        'name_en': M['nodes_df'].name_en.iloc[order[k]],
        'gen': M['gen'][order[k]],
        'flux_share': M['rho'][order[k]] / M['S'].sum(),
        'solo': 1 - delivered([order[k]]) / base,
        'cumulative': 1 - delivered(order[:k + 1]) / base,
    } for k in range(K)])

    pool = order[:pool_size]
    solo = pd.DataFrame([{'i': c, 'name_en': M['nodes_df'].name_en.iloc[c],
                          'gen': M['gen'][c], 'flux': M['rho'][c],
                          'solo': 1 - delivered([c]) / base} for c in pool])

    top5 = curve.head(5)
    joint = 1 - delivered(top5['i'].tolist()) / base
    agg = dict(base=float(base), n_eligible=len(elig), pool_size=len(pool),
               sum_solo=float(top5.solo.sum()), joint=float(joint),
               overlap=float(top5.solo.sum() - joint))
    return curve, solo, agg


# How much flux per generation
def concentration(M):
    layers, levels = _layers(M)
    def gini(x):
        x = np.sort(np.asarray(x, float))
        if x.sum() <= 0: return np.nan
        i = np.arange(1, len(x) + 1)
        return float((2 * i - len(x) - 1).dot(x) / (len(x) * x.sum()))
    out = []
    for t in levels:
        m = layers[t]
        if len(m) < 5: continue
        f = M['rho'][m]
        if f.sum() <= 0: continue
        o = np.sort(f)[::-1]
        out.append({'gen': t, 'n': len(m), 'gini': gini(f),
                    'top1': o[0] / f.sum(), 'top5': o[:5].sum() / f.sum()})
    return pd.DataFrame(out)


from scipy.stats import spearmanr

def inflow_by_index(M, v, top=6):
    df = M['nodes_df']
    F = edge_flux(M['P'], M['rho_full'], M['nodes'], M['name_map'])
    student = df.name_en.iloc[v]
    sub = F[F.student == student].copy()
    sub['share'] = sub.flux / sub.flux.sum()
    return student, sub.head(top)[['teacher', 'flux', 'share']]


def _route_counts(D, roots, terminal):
    """Routes root -> v -> collector through each node, on a DAG.

    up[v]   = 1 if v is a root, plus the sum of up[] over v's teachers
    down[v] = 1 if v hands material straight to the collector, plus the sum
              of down[] over v's students
    routes through v = up[v] * down[v]

    """
    order = list(nx.topological_sort(D))
    up = {v: 0.0 for v in D}
    down = {v: 0.0 for v in D}
    for v in order:
        up[v] = (1.0 if v in roots else 0.0) + sum(up[t] for t in D.predecessors(v))
    for v in reversed(order):
        down[v] = (1.0 if terminal.get(v, 0) > 0 else 0.0) + sum(down[s] for s in D.successors(v))
    return {v: up[v] * down[v] for v in D}


def centrality_baseline(M, abl, top=10, verbose=True):
    _, solo, _ = abl

    # Directed vertical graph, teacher -> student, weighted by multiplicity
    c = M['A_vert'].tocoo()
    D = nx.DiGraph()
    D.add_nodes_from(range(M['A_vert'].shape[0]))
    for t, s, w in zip(c.col, c.row, c.data):
        D.add_edge(int(t), int(s), weight=float(w))

    nodes = M['nodes']
    terminal = {i: M['terminal_w'].get(v, 0) for i, v in enumerate(nodes)}
    roots = set(int(r) for r in M['roots'])

    scores = {
        'Weighted degree': dict(D.degree(weight='weight')),
        'Betweenness (shortest path)': nx.betweenness_centrality(D, weight=None),
        'PageRank': nx.pagerank(D, weight='weight'),
        'Eigenvector': None,
        'Route count': _route_counts(D, roots, terminal),
        'Flux rho': {i: float(M['rho'][i]) for i in range(len(nodes))},
    }

    idx = solo['i'].tolist()
    sigma = solo['solo'].to_numpy()
    top_sigma = set(solo.nlargest(top, 'solo')['i'])

    rows = []
    for name, sc in scores.items():
        if sc is None:
            rows.append({'measure': name, 'spearman_r': np.nan, 'p': np.nan,
                         f'top{top}_overlap': np.nan, 'note': 'not defined on an acyclic graph'})
            continue
        vals = pd.Series([sc.get(i, 0.0) for i in idx], index=idx)
        r, p = spearmanr(vals, sigma)
        overlap = len(top_sigma & set(vals.nlargest(top).index))
        rows.append({'measure': name, 'spearman_r': r, 'p': p,
                     f'top{top}_overlap': overlap, 'note': ''})

    out = pd.DataFrame(rows)
    if verbose:
        print(out.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    return out

In [ ]:
_KNOWN = {
    'Muhammad ibn Shihab al-Zuhri': 'al-Zuhri',
    'Shu`ba ibn al-Hajjaj al-`Ataki': "Shu`ba",
    'Malik ibn Anas al-Asbahi': 'Malik',
    '`Urwa ibn Az-Zubayr al-Asadi': "`Urwa",
    'Sufyan ibn `Uyayna al-Hilali': 'Ibn `Uyayna',
    'Sufyan ath-Thawri': 'al-Thawri',
    'al-Layth ibn Sa`d al-Fhmi': 'al-Layth',
    'Qutayba ibn Sa`id al-Thaqafi': 'Qutayba',
    '`Ali ibn al-Madini': 'Ibn al-Madini',
    'Msdd ibn Msrhd al-Asadi': 'Musaddad',
    '`Abd Allah ibn Yusuf al-Kla`ai': 'Ibn Yusuf',
}

def short_name(full, seen=None):
    if full in _KNOWN:
        return _KNOWN[full]
    toks = str(full).split()
    nisba = next((t for t in reversed(toks) if t.lower().startswith(('al-', 'ath-', 'az-'))), None)
    if nisba and (seen is None or list(seen).count(nisba) <= 1):
        return nisba
    ism = toks[0] if toks else '?'
    return f"{ism} {nisba}" if nisba else ism

In [ ]:
# Most plots were generated with the help of AI, but have been checked for accuracy

def fig_ablation(M, abl, fname='fig_ablation.png', show=True):
    curve, solo, agg = abl
    conc = concentration(M)

    with plt.rc_context(_STYLE):
        fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))

        ax = axs[0]


        ax.plot(curve.k, curve.cumulative, 'o-', ms=4, color='#c44e52',
                label='cumulative')
        ax.plot(curve.k, curve.solo.cumsum(), '^:', ms=4, color='#888',
                label='sum of solo')
        ax.fill_between(curve.k, curve.cumulative, curve.solo.cumsum(),
                        alpha=.12, color='#c44e52')
        ax.legend(frameon=False, fontsize=7)



        ax.fill_between(curve.k, 0, curve.cumulative, alpha=.10, color='#c44e52')
        for r in curve.head(4).itertuples():
            ax.annotate(short_name(r.name_en), (r.k, r.cumulative),
                textcoords='offset points', xytext=(6, -9), fontsize=6.5)
        ax.set_xlabel('narrators removed ($k$)')
        ax.set_ylabel('fraction of corpus severed')
        ax.set_ylim(0, None)
        ax.set_xlim(left=0.5)
        ax.set_title('(a) cumulative severance', loc='left')
        ax.grid(alpha=.25)

        ax = axs[1]
        if not conc.empty:
            ax.plot(conc.gen + 1, conc.gini, 'o-', ms=4,
                    color='#8172b2', label='Gini')
            ax.plot(conc.gen + 1, conc.top5, 's--', ms=4,
                    color='#c44e52', label='top-5 share')
            ax.set_ylim(0, 1)
            ax.legend(frameon=False, fontsize=7, loc='upper left')
        ax.set_xlabel('\u0163abaqa')
        ax.set_ylabel('concentration of flux')
        ax.set_title('(b) where the corpus narrows', loc='left')
        ax.grid(alpha=.25)

        fig.tight_layout()
        if fname:
            fig.savefig(fname, bbox_inches='tight', dpi=300)
        if show:
            plt.show()
    return fig

In [ ]:
def uluww_table(M, top=25, min_share=1e-3):
    """Elevation index per narrator.

    skip = gen - mean_hops: generations bypassed on the way to v. Zero when
    every route traverses every intervening generation; positive when some
    material arrives by an elevated ('ali) chain that skips a link.

    p_elev is the share of v's material arriving strictly earlier than its
    modal hop -- the non-degenerate part of the depth profile, which the
    mean alone conceals."""
    RK, nodes, name_map, idx = M['RK'], M['nodes'], M['name_map'], M['idx']
    d = M['nodes_df'].copy()
    d = d[(d.flux_share >= min_share) & d.gen.notna() & d.mean_hops.notna()]
    d = d.nlargest(top, 'flux')

    cols = [idx[i] for i in d.narrator_id]
    Q = RK[:, cols]
    Q = Q / np.where(Q.sum(axis=0) > 0, Q.sum(axis=0), 1.0)

    k = np.arange(Q.shape[0])[:, None]
    mode = Q.argmax(axis=0)
    p_elev = np.array([Q[:m, j].sum() for j, m in enumerate(mode)])
    spread = np.sqrt(((k - Q.T.dot(k).T) ** 2 * Q).sum(axis=0))   # sd of profile
    n_routes = (Q > 0.01).sum(axis=0)

    out = pd.DataFrame({
        'name_en': [short_name(str(name_map.get(nodes[c], '?'))) for c in cols],
        'gen': d.gen.values, 'flux_share': d.flux_share.values,
        'mean_hops': d.mean_hops.values,
        'skip': d.gen.values - d.mean_hops.values,
        'mode_k': mode, 'p_elev': p_elev,
        'sd_hops': spread, 'n_routes': n_routes,
    })
    return out.sort_values('skip', ascending=False)

In [ ]:
COL = {'bukhari': '#c44e52', 'muslim': '#4c72b0'}
MRK = {'bukhari': 'o', 'muslim': 's'}
LBL = {'bukhari': 'Bukhārī', 'muslim': 'Muslim'}

# Display names as spelled in the paper; anything else falls back to short_name()
_DISPLAY = {'al-Zuhri': 'al-Zuhrī', "Shu`ba": 'Shuʿba', 'Malik': 'Mālik',
            "`Urwa": 'ʿUrwa', 'Ibn `Uyayna': 'Ibn ʿUyayna', 'al-Thawri': 'al-Thawrī',
            'al-Layth': 'al-Layth', 'Qutayba': 'Qutayba', 'Ibn al-Madini': 'Ibn al-Madīnī',
            'Musaddad': 'Musaddad', 'Ibn Yusuf': 'Ibn Yūsuf'}

def display_name(full):
    s = short_name(str(full))
    return _DISPLAY.get(s, s)


# ---------- Figure 1: transmission backbone ----------------------------
def fig_backbone(M, collection='Bukhārī', per_gen=6, n_edges=45, label_n=14,
                 highlight=(4, 7), fname='fig_backbone.pdf'):
    """Heaviest edges by flux between the per_gen heaviest narrators of each
    tabaqa. Node area ~ flux, edge width ~ edge flux. One-based tabaqat."""
    rho, gen, OMEGA, rho_full = M['rho'], M['gen'], M['OMEGA'], M['rho_full']
    layers, levels = _layers(M)
    X_OMEGA = float(np.nanmax(gen)) + 1.0
    XS = 2.6

    sel = set()
    for t in levels:
        m = layers[t]
        sel.update(int(v) for v in m[np.argsort(-rho[m])][:per_gen])

    F = M['P'].multiply(rho_full[np.newaxis, :]).tocoo()
    ed = sorted(((int(j), int(i), float(f)) for i, j, f in zip(F.row, F.col, F.data)
                 if int(j) in sel and (int(i) in sel or int(i) == OMEGA)),
                key=lambda e: -e[2])[:n_edges]

    cols = {}
    for v in sel | {v for e in ed for v in e[:2]}:
        x = X_OMEGA if v == OMEGA else gen[v]
        if not np.isnan(x):
            cols.setdefault(float(x), []).append(v)
    pos = {}
    for x, vs in cols.items():
        vs.sort(key=lambda v: -rho_full[v])
        for r, v in enumerate(vs):
            pos[v] = (x * XS, (r - (len(vs) - 1) / 2) * 1.25)

    narr = [v for v in pos if v != OMEGA]
    rmax = max(rho[v] for v in narr)
    labelled = set(sorted(narr, key=lambda v: -rho[v])[:label_n])
    cmap = plt.get_cmap('viridis')

    with plt.rc_context(_STYLE):
        fig, ax = plt.subplots(figsize=(10, 5.2))
        for t in highlight:                                  # bottleneck tabaqat
            ax.axvspan((t - 1) * XS - 0.9, (t - 1) * XS + 0.9, color='#f2e6c9', zorder=0, lw=0)
        for x in sorted(cols):
            ax.axvline(x * XS, color='#eeeeee', lw=0.8, zorder=0)

        mx = max(e[2] for e in ed)
        for a, b, f in ed:
            if a in pos and b in pos:
                (x0, y0), (x1, y1) = pos[a], pos[b]
                c = (x1 - x0) * 0.45
                ax.add_patch(mpatches.PathPatch(
                    mpath.Path([(x0, y0), (x0 + c, y0), (x1 - c, y1), (x1, y1)], [1, 4, 4, 4]),
                    facecolor='none', edgecolor='#5c7b99', lw=0.4 + 7 * f / mx,
                    alpha=max(0.2, 0.75 * f / mx), zorder=1, capstyle='round'))

        for v, (x, y) in pos.items():
            if v == OMEGA:
                ax.scatter(x, y, s=380, color='#222222', zorder=3)
                ax.text(x + 0.35, y, collection, va='center', fontsize=8, zorder=4)
                continue
            ax.scatter(x, y, s=20 + 330 * rho[v] / rmax, zorder=3, lw=0.6, edgecolor='white',
                       color=cmap(0.9 * gen[v] / X_OMEGA))
            if v in labelled:
                ax.text(x + 0.22, y, display_name(M['name_map'].get(M['nodes'][v], '?')),
                        fontsize=7, va='center', zorder=4,
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=0.8))

        ticks = sorted(cols)
        ax.set_xticks([x * XS for x in ticks])
        ax.set_xticklabels(['Ω' if x == X_OMEGA else str(int(x) + 1) for x in ticks])
        ax.set_xlabel('ṭabaqa')
        ax.set_yticks([])
        for s in ('top', 'right', 'left'):
            ax.spines[s].set_visible(False)
        ax.margins(x=0.04)
        fig.tight_layout()
        fig.savefig(fname, bbox_inches='tight')
    return fig


# ---------- Figure 2: redundancy ---------------------------------------
def fig_redundancy(obs, reps, fname='fig_redundancy.pdf'):
    """(a) sum of solo severance vs joint (cumulative) severance of the top k;
    the gap is redundancy. (b) redundancy / solo sum with 95% bootstrap bands;
    * marks k where the Muslim - Bukhari interval excludes zero."""
    diff = compare(reps['muslim'], reps['bukhari'])
    sig = diff.loc[((diff.lo > 0) | (diff.hi < 0)) & (diff.k >= 3), 'k']

    with plt.rc_context(_STYLE):
        fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))

        ax = axs[0]
        for name in obs:
            o = obs[name][0]
            c = COL[name]
            ax.plot(o.k, o.solo_sum, ls=':', marker=MRK[name], ms=3.5, color=c, alpha=.6)
            ax.plot(o.k, o.cumulative, ls='-', marker=MRK[name], ms=4, color=c, label=LBL[name])
            ax.fill_between(o.k, o.cumulative, o.solo_sum, color=c, alpha=.12, lw=0)
        ax.plot([], [], ls=':', color='#777', label='sum of solo')
        ax.plot([], [], ls='-', color='#777', label='joint (cumulative)')
        ax.set_xlabel('narrators removed ($k$)')
        ax.set_ylabel('share of delivered flow severed')
        ax.set_ylim(0, None)
        ax.legend(frameon=False, fontsize=8, loc='upper left')
        ax.set_title('(a)', loc='left')

        ax = axs[1]
        for name in obs:
            o = obs[name][0]; o = o[o.k >= 3]
            q = reps[name]['curve'].groupby('k')['share'].quantile([.025, .975]).unstack()
            q = q.loc[q.index >= 3]
            ax.fill_between(q.index, q[0.025], q[0.975], color=COL[name], alpha=.18, lw=0)
            ax.plot(o.k, o.share, marker=MRK[name], ms=4, color=COL[name], label=LBL[name])
        top = ax.get_ylim()[1] * 1.10                   # headroom for the stars
        for k in sig:
            ax.text(k, top * 0.99, '*', ha='center', va='top', fontsize=12)
        ax.set_xlabel('narrators removed ($k$)')
        ax.set_ylabel('redundancy / sum of solo')
        ax.set_ylim(0, top)
        ax.legend(frameon=False, fontsize=8, loc='lower right')
        ax.set_title('(b)', loc='left')

        kmax = int(obs['bukhari'][0].k.max())
        axs[0].set_xticks(range(1, kmax + 1))
        axs[1].set_xticks(range(3, kmax + 1))
        for ax in axs:
            ax.grid(alpha=.25)
        fig.tight_layout()
        fig.savefig(fname, bbox_inches='tight')
    return fig


# ---------- Figure 3: concentration by tabaqa --------------------------
def fig_gini(obs, peaks=(4, 7), n_min=20, fname='fig_gini.pdf'):
    """Gini of flux within each tabaqa, both collections. Shaded: the peaks
    tested in Table tab:peaks. Hollow markers: cohorts with fewer than n_min
    narrators."""
    with plt.rc_context(_STYLE):
        fig, ax = plt.subplots(figsize=(6.2, 3.6))
        for t in peaks:
            ax.axvspan(t - 0.4, t + 0.4, color='#f2e6c9', lw=0, zorder=0)
        for name in obs:
            g = obs[name][1].assign(t=lambda d: d.gen.astype(int) + 1)
            c = COL[name]
            ax.plot(g.t, g.gini, '-', marker=MRK[name], ms=5, color=c, lw=1.3,
                    label=LBL[name], zorder=2)
            small = g.n < n_min                      # hollow: small cohort
            ax.scatter(g.t[small], g.gini[small], marker=MRK[name], s=26,
                       facecolor='white', edgecolor=c, zorder=3)
        ax.set_xlabel('ṭabaqa')
        ax.set_ylabel('Gini coefficient of flux')
        ax.set_ylim(0, 1)
        ax.set_xticks(range(1, int(max(o[1].gen.max() for o in obs.values())) + 2))
        ax.grid(alpha=.25)
        ax.legend(frameon=False, fontsize=8, loc='lower right')
        fig.tight_layout()
        fig.savefig(fname, bbox_inches='tight')
    return fig

In [ ]:
if __name__ == '__main__':
    print("Building model...")
    M = build_model()
    df, d = M['nodes_df'], M['diag']

    print(f"\n--- NETWORK DIAGNOSTICS ---")
    print(f"Nodes: {d['n_nodes']} | Vertical edges: {d['n_vertical_edges']}")
    print(f"Longest chain: {d['longest_chain']} hops")
    print(f"Roots: {d['n_roots']}  "
          f"(tabaqa 1: {d['roots_gen0']}, "
          f"max tabaqa: {d['roots_gen_max'] + 1:.0f}, "
          f"no generation: {d['roots_no_gen']})")

    print(f"\n--- PREPROCESSING DROPS (share of recorded transmissions) ---")
    print(f"Kept (vertical):            {d['kept_share']:.2%}")
    print(f"Lateral (same tabaqa):      {d['lateral_share']:.2%}")
    print(f"Over gen cap (>{d['gen_cap']}):        {d['over_gen_cap_share']:.2%}")
    print(f"Missing generation:         {d['no_generation_share']:.2%}")

    print(f"\n--- CONSERVATION ---")
    print(f"Injected:  {d['injected']:.1f}")
    print(f"Absorbed:  {d['absorbed']:.1f} ({d['absorbed']/d['injected']:.2%})")
    print(f"Stranded:  {d['stranded']:.1f} ({d['orphan_share']:.2%})")
    print(f"Unreached nodes: {d['unreached']}")
    print(f"Mean chain length: model {d['mean_chain_len_model']:.2f} hops | "
          f"empirical {d['mean_chain_len_empirical']:.2f}")

    print(f"Roots: {d['roots_gen0']: .1f}")

    print(f"\n--- ABLATION (VULNERABILITY) ---")
    abl = ablation(M)
    curve, solo, agg = abl
    print(curve.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"Top-5 solo sum: {agg['sum_solo']:.3f} | jointly: {agg['joint']:.3f}")
    print(concentration(M).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(curve.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"\nTop-5 individually sever: {agg['sum_solo']:.3f} (Sum)")
    print(f"Top-5 jointly sever:      {agg['joint']:.3f} (Overlap)")

    print(f"\n--- CONCENTRATION ---")
    print(concentration(M).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print(f"\n--- ULUWW ---")
    print(uluww_table(M).to_string(index=False,
      float_format=lambda x: f"{x:.3f}"))

    print(f"\n--- CENTRALITY CORRELATION ---")
    centrality_baseline(M, abl)


    print("\nGenerating Figures...")
    fig_ablation(M, abl)
    fig_backbone(M)

In [ ]:
raw_in = Counter()
for (a, b), w in M['edge_weights'].items():
    raw_in[b] += w

rt = pd.DataFrame({
    'idx':     M['roots'],
    'nid':     [M['nodes'][r] for r in M['roots']],
    'name_en': [M['name_map'].get(M['nodes'][r], '?') for r in M['roots']],
    'gen':     M['gen'][M['roots']] + 1,
    'raw_in':  [raw_in.get(M['nodes'][r], 0) for r in M['roots']],
    'flux':    M['rho'][M['roots']],
})
rt['artefact'] = rt.raw_in > 0

print(pd.crosstab(rt.gen, rt.artefact))
print(f"artefact roots: {rt.artefact.sum()} / {len(rt)} "
      f"({rt.artefact.mean():.1%} of injected mass)")

In [ ]:
print(rt[~rt.artefact & (rt.gen >= 1)].sort_values('flux', ascending=False).head(15).to_string(index=False))

In [ ]:
# ----------------------------------------------------------------------
# BOOTSTRAP OVER TRADITIONS
# ----------------------------------------------------------------------
# Uses build_model, ablation and concentration from the model cell unchanged.


def _pct(x, q=(2.5, 97.5)):
    """Percentiles of x, ignoring NaN (linear interpolation, as numpy/pandas)."""
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    return tuple(np.percentile(x, q)) if len(x) else (np.nan,) * len(q)


def _delivered(M_, removed=()):
    """Flow reaching the collector with the narrators in `removed` deleted
    (the same solve as inside ablation())."""
    P, S, N = M_['P'], M_['S'], M_['P'].shape[0]
    keep = np.ones(N)
    keep[list(removed)] = 0.0
    D = sp.diags(keep)
    return spla.spsolve((sp.eye(N, format='csc') - D @ P @ D).tocsc(), S * keep)[M_['OMEGA']]


def statistics(M_, K=12, track=None):
    """Every statistic the bootstrap follows, for one model (observed or resampled).

    curve   redundancy among the top-k narrators, k = 1..K, re-ranked by this
            model's flux: solo_sum - cumulative, and its share of solo_sum
    gini    Gini coefficient of flux per tabaqa (gen is zero-based); used for
            the observed collection only (Table tab:gini, fig_gini)
    overlap redundancy of fixed narrator sets given by narrator_id, e.g. a
            teacher and his relay: sum of solo severances minus joint severance
    """
    curve, _, _ = ablation(M_, K=K, pool_size=0)
    curve['solo_sum'] = curve['solo'].cumsum()
    curve['redundancy'] = curve['solo_sum'] - curve['cumulative']
    curve['share'] = curve['redundancy'] / curve['solo_sum']

    base, overlap = _delivered(M_), {}
    for label, ids in (track or {}).items():
        ix = [M_['idx'].get(v) for v in ids]
        if any(i is None for i in ix):              # absent from this replicate
            overlap[label] = np.nan
            continue
        solo = sum(1 - _delivered(M_, [i]) / base for i in ix)
        overlap[label] = float(solo - (1 - _delivered(M_, ix) / base))

    return curve, concentration(M_)[['gen', 'n', 'gini']], overlap


def bootstrap(df, meta, B=1000, K=12, seed=0, track=None, progress=100):
    """Draw len(df) traditions with replacement, each with all its chains,
    rebuild the model with build_model unchanged, and recompute statistics().
    Returns the replicates; the observed values come from statistics() on the
    full collection."""
    rng = np.random.default_rng(seed)
    T = len(df)
    curves, overlaps = [], []
    for b in range(B):
        M_ = build_model(df=df.iloc[rng.integers(0, T, T)], meta=meta)
        c, _, o = statistics(M_, K, track)
        curves.append(c.assign(b=b))
        overlaps.append(dict(o, b=b))
        if progress and (b + 1) % progress == 0:
            print(f'  replicate {b + 1}/{B}')
    return dict(curve=pd.concat(curves, ignore_index=True),
                track=pd.DataFrame(overlaps))


def resolve_ids(meta, M_, patterns):
    """narrator_id for each name pattern: of the narrators whose name contains
    it, the one appearing most often in this collection's edges. Check the
    printed matches."""
    present = Counter()
    for (a, b), w in M_['edge_weights'].items():
        present[a] += w
        present[b] += w
    names = meta.set_index('narrator_id')['name_en']
    out = []
    for pat in patterns:
        hits = [v for v in meta.narrator_id[meta.name_en.astype(str).str.contains(pat, regex=False)]
                if v in present]
        if not hits:
            raise KeyError(f'no narrator matching {pat!r} in this collection')
        best = max(hits, key=lambda v: present[v])
        print(f'  {pat!r:40s} -> {best}  {names[best]}')
        out.append(best)
    return out


# ---------- comparisons and tables --------------------------------------
def compare(reps_a, reps_b, col='share'):
    """Interval of (a - b) per k. The collections are resampled independently;
    replicates are paired by index. Excluding zero means the difference
    survives resampling."""
    a = reps_a['curve'].pivot(index='b', columns='k', values=col)
    b = reps_b['curve'].pivot(index='b', columns='k', values=col)
    d = (a - b).dropna(how='all')
    return pd.DataFrame([dict(k=k, lo=_pct(d[k])[0], median=np.nanmedian(d[k]), hi=_pct(d[k])[1])
                         for k in d.columns])


def redundancy_table(obs, reps, a='muslim', b='bukhari', k_min=3):
    """Table tab:redundancy: relative redundancy of the top k in each
    collection with 95% percentile intervals, and the difference a - b."""
    diff = compare(reps[a], reps[b]).set_index('k')
    rows = []
    for k in obs[b][0]['k']:
        if k < k_min:
            continue
        row = {'k': int(k)}
        for name in (b, a):
            observed = obs[name][0].set_index('k').loc[k, 'share']
            lo, hi = _pct(reps[name]['curve'].loc[lambda d: d.k == k, 'share'])
            row.update({name: observed, f'{name}_lo': lo, f'{name}_hi': hi})
        lo, hi = diff.loc[k, 'lo'], diff.loc[k, 'hi']
        row.update(diff_median=diff.loc[k, 'median'], diff_lo=lo, diff_hi=hi,
                   excludes_zero=bool(lo > 0 or hi < 0))
        rows.append(row)
    return pd.DataFrame(rows)


def overlap_table(obs, reps):
    """Redundancy of the tracked narrator sets, as a share of delivered flow,
    with 95% percentile intervals."""
    rows = []
    for name in obs:
        for label, observed in obs[name][2].items():
            lo, hi = _pct(reps[name]['track'][label])
            rows.append(dict(collection=name, set=label, observed=observed, lo=lo, hi=hi))
    return pd.DataFrame(rows)


# ---------- figure -------------------------------------------------------
def fig_redundancy_ci(obs, reps, fname='fig_redundancy_ci.pdf', col='share', k_min=3):
    """Relative redundancy across k with 95% bootstrap bands, per collection."""
    colours = {'bukhari': '#c44e52', 'muslim': '#4c72b0'}
    labels = {'bukhari': 'Bukhārī', 'muslim': 'Muslim'}
    with plt.rc_context(_STYLE):
        fig, ax = plt.subplots(figsize=(6, 4))
        for name in obs:
            o = obs[name][0].loc[lambda d: d.k >= k_min]
            band = (reps[name]['curve'].loc[lambda d: d.k >= k_min]
                    .groupby('k')[col].apply(lambda x: pd.Series(_pct(x), index=['lo', 'hi']))
                    .unstack())
            ax.fill_between(band.index, band.lo, band.hi, alpha=.18, color=colours.get(name), lw=0)
            ax.plot(o.k, o[col], 'o-', ms=4, color=colours.get(name), label=labels.get(name, name))
        ax.set_xlabel('narrators removed ($k$)')
        ax.set_ylabel('redundancy / sum of solo severance' if col == 'share'
                      else 'redundancy (share of delivered flow)')
        ax.set_ylim(0, None)
        ax.legend(frameon=False)
        ax.grid(alpha=.25)
        fig.tight_layout()
        fig.savefig(fname, bbox_inches='tight')
    return fig

In [ ]:
B_REPS = 1000
K_TOP  = 12
COLLECTIONS = {'bukhari': 'bukhari.json', 'muslim': 'muslim.json'}

TRACK_NAMES = {'Shu`ba + Ghundar': ['Shu`ba ibn al-Hajjaj', 'Muhammad ibn Ja`far']}

meta_bs = build_metadata()
bs_obs, bs_reps = {}, {}
for seed, (name, path) in enumerate(COLLECTIONS.items(), start=1):
    print(f'\n--- BOOTSTRAP: {name} ---')
    df_c, _, _ = load_chains(meta_bs, path)
    M_c = build_model(df=df_c, meta=meta_bs)
    track = {lbl: resolve_ids(meta_bs, M_c, pats) for lbl, pats in TRACK_NAMES.items()}
    bs_obs[name] = statistics(M_c, K_TOP, track)
    bs_reps[name] = bootstrap(df_c, meta_bs, B=B_REPS, K=K_TOP, seed=seed, track=track)

# Keep the replicates, so the tables can be redone without rerunning:
# bs_obs, bs_reps = pd.read_pickle('bootstrap_results.pkl')
pd.to_pickle((bs_obs, bs_reps), 'bootstrap_results.pkl')

In [ ]:
fmt = lambda x: f'{x:.3f}'

print('--- Redundancy among the top k, relative to the solo sum (Table tab:redundancy) ---')
print(redundancy_table(bs_obs, bs_reps).to_string(index=False, float_format=fmt))

print('\n--- Tracked sets, share of delivered flow ---')
print(overlap_table(bs_obs, bs_reps).to_string(index=False, float_format=fmt))

for name in COLLECTIONS:
    print(f'\n--- {name}: Gini of flux per tabaqa (Table tab:gini) ---')
    print(bs_obs[name][1].assign(tabaqa=lambda d: d.gen.astype(int) + 1)[['tabaqa', 'n', 'gini']]
          .to_string(index=False, float_format=fmt))

fig_redundancy_ci(bs_obs, bs_reps)

## Robustness

Reruns the headline numbers under alternative modelling choices: seeding only the Companions (so roots created by the filters inject nothing), out-strength seeding, and generation caps of 3 and 5. `rank agreement` is the Spearman correlation of flux with the baseline over the baseline's sixty heaviest non-root narrators. `gaps` reports how many traditions contain a null (an unidentified narrator). The mean chain under out-strength seeding is the value reported as $\bar\ell$ in Table 1.

In [ ]:
# ----------------------------------------------------------------------
# ROBUSTNESS: SEEDING AND GENERATION CAP
# ----------------------------------------------------------------------
# Reruns the headline numbers under alternative modelling choices. Uses
# build_model, solve_flux and statistics() unchanged; only the source vector
# or the generation cap differs.

# Same as before
def _reseed(M, S_new):
    M = dict(M)
    rho_full = solve_flux(M['P'], S_new)
    M.update(S=S_new, rho_full=rho_full, rho=rho_full[:len(M['nodes'])],
             absorbed=rho_full[M['OMEGA']])
    return M

# Only seed the tabaqa 1
def companion_only(M):
    S = M['S'].copy()
    S[[r for r in M['roots'] if M['gen'][r] != 0]] = 0.0
    return _reseed(M, S)

# How often a chain contains a null (an unidentified narrator).

def gap_report(df):
    idx = df['transmission_information'].map(lambda t: (t or {}).get('scholar_indices') or [])
    has_null = idx.map(lambda xs: any(x is None or x == '' for x in xs))
    n_seg = df['segments'].map(len)
    return dict(traditions=len(df), segments=int(n_seg.sum()),
                with_null=float(has_null.mean()), multi_segment=float((n_seg > 1).mean()))


def variant_row(M, K, track, pool_ids, base_rho):
    """Headline numbers for one model variant."""
    curve, gini, overlap = statistics(M, K, track)
    share = curve.set_index('k')['share']
    g = gini.assign(t=gini.gen.astype(int) + 1).set_index('t')['gini']
    rho = pd.Series(M['rho'], index=M['nodes'])
    S = M['S'].sum()
    row = {
        'delivered': M['rho_full'][M['OMEGA']] / S,
        'mean chain (narrators)': float(M['nu'] @ M['S']) / S - 1.0,
        'rank agreement (r_s, top 60)': spearmanr(base_rho.loc[pool_ids], rho.loc[pool_ids])[0],
        'top 5': '; '.join(n[:20] for n in curve.name_en.head(5)),
        'top-1 solo': curve.solo.iloc[0],
        **{f'redundancy k={k}': share.get(k, np.nan) for k in (3, 5, 7)},
        **{f'overlap {lbl}': v for lbl, v in overlap.items()},
        'G4 - G3': g.get(4) - g.get(3), 'G4 - G5': g.get(4) - g.get(5),
        'G7 - G6': g.get(7) - g.get(6), 'G7 - G8': g.get(7) - g.get(8),
    }
    return row


def robustness(path, meta, track, K=12, caps=(3, 5)):
    """Rows: baseline (uniform, cap GEN_CAP), Companion-only seeding,
    out-strength seeding, and the generation caps in `caps`."""
    global GEN_CAP
    base_cap = GEN_CAP
    M0 = build_model(path=path, meta=meta)
    base_rho = pd.Series(M0['rho'], index=M0['nodes'])
    roots = set(M0['nodes'][r] for r in M0['roots'])
    pool_ids = [v for v in base_rho.sort_values(ascending=False).index if v not in roots][:60]

    variants = {f'baseline (uniform, cap {base_cap})': M0,
                'Companion-only seeding': companion_only(M0),
                'out-strength seeding': build_model(seeding='outstrength', path=path, meta=meta)}
    try:
        for cap in caps:
            GEN_CAP = cap
            variants[f'cap {cap}'] = build_model(path=path, meta=meta)
    finally:
        GEN_CAP = base_cap
    return pd.DataFrame({name: variant_row(M, K, track, pool_ids, base_rho)
                         for name, M in variants.items()})


rob = {}
for name, path in COLLECTIONS.items():
    print(f'\n=== ROBUSTNESS: {name} ===')
    df_c, _, _ = load_chains(meta_bs, path)
    print('gaps:', {k: round(v, 4) if isinstance(v, float) else v for k, v in gap_report(df_c).items()})
    M_c = build_model(df=df_c, meta=meta_bs)
    track = {lbl: resolve_ids(meta_bs, M_c, pats) for lbl, pats in TRACK_NAMES.items()}
    rob[name] = robustness(path, meta_bs, track, K=K_TOP)
    print(rob[name].drop('top 5').to_string(float_format=lambda x: f'{x:.3f}'))
    print('top 5 by flux:')
    for variant, names in rob[name].loc['top 5'].items():
        print(f'  {variant:28s} {names}')

In [ ]:
M_bukhari = build_model(path=COLLECTIONS['bukhari'])
fig_backbone(M_bukhari, collection='Bukhārī', fname='fig_backbone.png')
fig_redundancy(bs_obs, bs_reps, fname='fig_redundancy.png')
fig_gini(bs_obs, fname='fig_gini.png')